# NOVA 03 — MOD-05 Face Embedding (MobileFaceNet)
**Attach datasets:**
- `yakhyokhuja/vggface2-112x112` — VGGFace2 pre-aligned to 112x112 (exactly MobileFaceNet's input size; no alignment preprocessing needed)
- `jessicali9530/lfw-dataset` (evaluation only)

**Accelerator:** GPU. Training on a subset of identities ~6-10h.
Teacher (InsightFace ArcFace) downloads its weights on first run — internet must be ON in notebook settings.

In [1]:
# ── NOVA bootstrap: clone repo + resolve HF token from Kaggle secret ──
# Before running: Add-ons > Secrets > attach a secret named HF_TOKEN
# (your HuggingFace write token). Settings > Accelerator > GPU T4/P100.
import os, sys, subprocess

REPO = 'https://github.com/BertinAm/nova-ml.git'
if not os.path.exists('/kaggle/working/nova-ml'):
    subprocess.run(['git', 'clone', REPO, '/kaggle/working/nova-ml'], check=True)
else:  # already cloned in this session — pull latest fixes
    subprocess.run(['git', '-C', '/kaggle/working/nova-ml', 'pull'], check=True)
os.chdir('/kaggle/working/nova-ml')
sys.path.insert(0, '/kaggle/working/nova-ml/scripts')

from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['NOVA_HF_USERNAME'] = 'unixio'

# GPU compatibility guard: Kaggle's PyTorch 2.10 image dropped sm_60 (P100).
# If you get a P100, switch Settings > Accelerator to 'GPU T4 x2'.
import torch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    assert cap >= (7, 0), (
        f'{name} (sm_{cap[0]}{cap[1]}) is NOT supported by this PyTorch build. '
        'Switch Settings > Accelerator to GPU T4 x2 and restart.')
    print(f'GPU OK: {name}')
else:
    raise RuntimeError('No GPU — set Settings > Accelerator to GPU T4 x2.')
print('Bootstrap OK — repo cloned, HF token loaded.')

Cloning into '/kaggle/working/nova-ml'...


GPU OK: Tesla T4
Bootstrap OK — repo cloned, HF token loaded.


In [2]:
!pip install -q insightface onnxruntime-gpu onnx2tf onnx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 762.2/762.2 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.3/220.3 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/6

In [3]:
# Auto-resolve mounts (Kaggle nests datasets 3 levels deep) and locate the
# identity-folder root: the directory whose children are identity folders.
import glob, os
inputs = (glob.glob('/kaggle/input/*') + glob.glob('/kaggle/input/*/*')
          + glob.glob('/kaggle/input/*/*/*'))
VGG_ROOT = next(p for p in inputs if 'vggface' in p.split('/')[-1].lower())
print('VGG_ROOT =', VGG_ROOT)
!find {VGG_ROOT} -maxdepth 2 -type d | head -10
# Pick the deepest dir that contains many subfolders (identities)
VGG_DATA = None
for cand in [VGG_ROOT] + glob.glob(f'{VGG_ROOT}/*') + glob.glob(f'{VGG_ROOT}/*/*'):
    if os.path.isdir(cand):
        subs = [s for s in os.listdir(cand)[:50]
                if os.path.isdir(os.path.join(cand, s))]
        if len(subs) >= 40:  # many identity folders
            VGG_DATA = cand
            break
assert VGG_DATA, 'Could not locate identity-folder root — inspect the layout above'
print('VGG_DATA =', VGG_DATA, '| identities (sample count):',
      len(os.listdir(VGG_DATA)))

VGG_ROOT = /kaggle/input/datasets/yakhyokhuja/vggface2-112x112
/kaggle/input/datasets/yakhyokhuja/vggface2-112x112
/kaggle/input/datasets/yakhyokhuja/vggface2-112x112/vggface2_112x112
/kaggle/input/datasets/yakhyokhuja/vggface2-112x112/vggface2_112x112/id_3050
/kaggle/input/datasets/yakhyokhuja/vggface2-112x112/vggface2_112x112/id_6622
/kaggle/input/datasets/yakhyokhuja/vggface2-112x112/vggface2_112x112/id_7752
/kaggle/input/datasets/yakhyokhuja/vggface2-112x112/vggface2_112x112/id_8321
/kaggle/input/datasets/yakhyokhuja/vggface2-112x112/vggface2_112x112/id_5671
/kaggle/input/datasets/yakhyokhuja/vggface2-112x112/vggface2_112x112/id_6982
/kaggle/input/datasets/yakhyokhuja/vggface2-112x112/vggface2_112x112/id_3251
/kaggle/input/datasets/yakhyokhuja/vggface2-112x112/vggface2_112x112/id_8229
find: ‘standard output’: Broken pipe
find: write error
VGG_DATA = /kaggle/input/datasets/yakhyokhuja/vggface2-112x112/vggface2_112x112 | identities (sample count): 8631


In [4]:
# FORCE RETRAIN: the checkpoint currently on HF (v1.0.0) is a known-bad
# collapsed model — LFW accuracy 0.500 (random). Do NOT reuse it. Once a
# genuinely good checkpoint is published, flip this back to True to save
# ~90 min on conversion-only re-runs.
SKIP_TRAINING = False
print('Forcing full retrain — HF checkpoint is the known-collapsed v1.0.0 model.')

Forcing full retrain — HF checkpoint is the known-collapsed v1.0.0 model.


In [5]:
# MobileFaceNet + ArcFace loss. --max-identities 2000 keeps one run inside
# Kaggle's 12h session limit. --no-teacher: InsightFace teacher only runs
# on CPU here (onnxruntime shadows the GPU build), ~10x slower, AND two
# earlier runs proved the teacher wasn't even the cause of the training
# collapse we hit (see ArcFaceLoss docstring — it was a missing easy-margin
# safeguard, now fixed) — so there is no upside to paying for it.
if not SKIP_TRAINING:
    !python scripts/train_face_embedding.py --data-dir {VGG_DATA} \
        --epochs 30 --batch-size 256 --max-identities 2000 \
        --max-per-identity 40 --no-teacher
else:
    print('Skipped — using HF checkpoint.')

Identities: 2000, images: 80000 (capped at 40/identity)
  epoch 1 step 0/312 loss 42.2846
  epoch 1 step 200/312 loss 39.6577
epoch 1/30 | loss 40.2616
  epoch 2 step 0/312 loss 38.4254
  epoch 2 step 200/312 loss 35.2877
epoch 2/30 | loss 36.1279
  epoch 3 step 0/312 loss 30.5021
  epoch 3 step 200/312 loss 29.0961
epoch 3/30 | loss 29.7806
  epoch 4 step 0/312 loss 24.7378
  epoch 4 step 200/312 loss 25.1852
epoch 4/30 | loss 25.8976
  epoch 5 step 0/312 loss 23.9659
  epoch 5 step 200/312 loss 24.3693
epoch 5/30 | loss 23.6705
  epoch 6 step 0/312 loss 21.0172
  epoch 6 step 200/312 loss 22.2152
epoch 6/30 | loss 22.2382
  epoch 7 step 0/312 loss 21.2516
  epoch 7 step 200/312 loss 20.3301
epoch 7/30 | loss 21.1466
  epoch 8 step 0/312 loss 19.9258
  epoch 8 step 200/312 loss 21.6334
epoch 8/30 | loss 20.3383
  epoch 9 step 0/312 loss 17.8075
  epoch 9 step 200/312 loss 20.4350
epoch 9/30 | loss 19.6496
  epoch 10 step 0/312 loss 17.9500
  epoch 10 step 200/312 loss 19.8687
epoch 10

In [6]:
# Build a SMALL calibration dir (200 images). Never point the converter at
# the full dataset — rglob over 3M files takes forever.
import os, random, shutil
CALIB = '/kaggle/working/calib'
shutil.rmtree(CALIB, ignore_errors=True)
os.makedirs(CALIB)
rng = random.Random(42)
idents = rng.sample(sorted(os.listdir(VGG_DATA)), 200)
for i, ident in enumerate(idents):
    d = os.path.join(VGG_DATA, ident)
    imgs = [f for f in os.listdir(d) if f.lower().endswith(('.jpg', '.png'))]
    if imgs:
        shutil.copy(os.path.join(d, imgs[0]), f'{CALIB}/{i:03d}_{imgs[0]}')
print(len(os.listdir(CALIB)), 'calibration images')

200 calibration images


In [7]:
# Convert: tries INT8 first, falls back to float32 (never ship onnx2tf's
# fp16 — true fp16 tensors fail to load on standard TFLite runtimes).
!python scripts/convert_to_tflite.py \
    --checkpoint /kaggle/working/checkpoints/face_embedding_best.pth \
    --arch mobilefacenet --input-size 112 \
    --out /kaggle/working/exports/face_embedding_v1.tflite \
    --calib-dir {CALIB} --benchmark

W0712 04:27:27.741000 630 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0712 04:27:28.515000 630 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0712 04:27:28.515000 630 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, 

In [8]:
# LFW verification (FR-05-05 acceptance metric: accuracy >= 99% target).
# Non-fatal: if the LFW mirror's CSV layout differs, publish proceeds and
# the eval can be added in a later 5-min checkpoint-reuse run.
LFW_ROOT = next((p for p in inputs if 'lfw' in p.split('/')[-1].lower()), None)
print('LFW_ROOT =', LFW_ROOT)
if LFW_ROOT:
    !python scripts/evaluate_lfw.py \
        --checkpoint /kaggle/working/checkpoints/face_embedding_best.pth \
        --lfw-root {LFW_ROOT} \
        --out /kaggle/working/evaluation/lfw_results.json || echo 'LFW eval failed — continuing'
else:
    print('LFW dataset not attached — skipping eval')

LFW_ROOT = /kaggle/input/datasets/jessicali9530/lfw-dataset
LFW pairs: 3200 (1600 same / 1600 different)
{
  "lfw_pairs": 3200,
  "accuracy_at_threshold_0.75": 0.5859375,
  "best_accuracy": 0.7690625,
  "best_threshold": 0.575,
  "tar_at_far_1e-3": 0.120625
}


In [9]:
# Publish to HuggingFace: unixio/nova-face-embedding
!python scripts/push_to_huggingface.py --module MOD-05-embed \
    --pytorch /kaggle/working/checkpoints/face_embedding_best.pth \
    --tflite /kaggle/working/exports/face_embedding_v1.tflite \
    --eval-json /kaggle/working/evaluation/lfw_results.json --version 1.0.0

Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  .../face_embedding_v1.tflite:  92%|████████████▊ | 4.39MB / 4.79MB            

Processing Files (0 / 1)      :  92%|████████████▊ | 4.39MB / 4.79MB, 21.9MB/s  
New Data Upload               :  92%|████████████▊ | 4.39MB / 4.79MB, 21.9MB/s  

Processing Files (1 / 1)      : 100%|██████████████| 4.79MB / 4.79MB, 12.0MB/s  
New Data Upload               : 100%|██████████████| 4.79MB / 4.79MB, 12.0MB/s  

  .../face_embedding_v1.tflite: 100%|██████████████| 4.79MB / 4.79MB            

Processing Files (1 / 1)      : 100%|██████████████| 4.79MB / 4.79MB, 7.98MB/s  
New Data Upload               : 100%|██████████████| 4.79MB / 4.79MB, 7.98MB/s  
  .../face_embedding_v1.tflite: 100%|██████████████| 4.79MB / 4.79MB            
Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload        